## Introduction

(written by the author)

## Imports and paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

DATA = Path("data")
TRAIN = DATA / "train_FD001.txt"
TEST = DATA / "test_FD001.txt"
RUL = DATA / "RUL_FD001.txt"

## Load data

In [ ]:
COLS = ["unit", "cycle"] + [f"setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]

train = pd.read_csv(TRAIN, sep=r"\s+", header=None, names=COLS)
test = pd.read_csv(TEST, sep=r"\s+", header=None, names=COLS)
rul = pd.read_csv(RUL, sep=r"\s+", header=None, names=["RUL"]).squeeze()

## Sanity check

`data/readme.txt` numbers its column list up to "sensor measurement 26", but the file has only 26 columns in total — unit + cycle + 3 operational settings + 21 sensors — so 21 sensors is the correct count and nothing was truncated.

In [ ]:
for name, df in [("train", train), ("test", test)]:
    print(f"{name}: shape={df.shape} engines={df.unit.nunique()} missing={int(df.isna().sum().sum())}")
print(f"rul: shape={rul.shape} missing={int(rul.isna().sum())}")
assert len(rul) == test["unit"].nunique() == 100
print("dtypes:", train.dtypes.value_counts().to_dict())
print(train.head(5).to_string())
print(rul.head(5).to_string())

## Train RUL

In [ ]:
train["RUL"] = train.groupby("unit")["cycle"].transform("max") - train["cycle"]
print(train.sample(5, random_state=0).to_string())
print(train["RUL"].describe().to_string())

## Sensor variability

In [ ]:
sensors = [c for c in train.columns if c.startswith("sensor_")]
nuniq = train[sensors].nunique()
print(nuniq.to_frame("distinct").T.to_string())

dropped = [s for s in sensors if nuniq[s] <= 5]  # flat: no trend to learn
KEEP_SENSORS = [s for s in sensors if s not in dropped]
print("dropped flat sensors (<= 5 distinct values):", dropped)
print("kept sensors:", len(KEEP_SENSORS))


## Degradation by engine

In [ ]:
corr = train[KEEP_SENSORS].apply(lambda s: s.corr(train["RUL"]))
TOP = list(corr.abs().sort_values(ascending=False).head(4).index)
life = train.groupby("unit")["cycle"].max()
engines = [life.idxmin(), (life - life.quantile(0.25)).abs().idxmin(),
           (life - life.quantile(0.75)).abs().idxmin(), life.idxmax()]

# One fixed colour per sensor, keyed to TOP in the same order; the fleet-trend
# cell reuses SENSOR_COLORS so a sensor keeps the same hue in both figures
# (Okabe-Ito, colourblind-safe).
SENSOR_COLORS = dict(zip(TOP, ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]))
ENGINE_DASH = ["-", (0, (5, 1.6)), (0, (1, 1.1)), (0, (6, 1.6, 1, 1.6))]
WIN = 15        # cycles in the centred rolling mean
MIN_ALIVE = 20  # fleet mean is drawn only while at least this many engines run

plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 10.5, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
    "axes.grid": True, "grid.color": "#E9E9E9", "grid.linewidth": 0.8,
    "axes.edgecolor": "#CCCCCC", "axes.spines.top": False,
    "axes.spines.right": False, "figure.facecolor": "white",
})

fig, axes = plt.subplots(2, 2, figsize=(11, 7.9), dpi=120, sharex=True)
fig.subplots_adjust(left=0.077, right=0.985, top=0.878, bottom=0.185,
                    hspace=0.40, wspace=0.19)
alive = train.groupby("cycle")["unit"].nunique()  # engines still recording per cycle

for ax, s in zip(axes.ravel(), TOP):
    col = SENSOR_COLORS[s]
    fleet = train.groupby("cycle")[s].mean()
    fleet = fleet[alive >= MIN_ALIVE].rolling(WIN, center=True, min_periods=3).mean()
    ax.plot(fleet.index, fleet.to_numpy(), color="#A6A6A6", lw=2.6, zorder=1)
    for u, dash in zip(engines, ENGINE_DASH):
        d = train.loc[train["unit"] == u].sort_values("cycle")
        smooth = d[s].rolling(WIN, center=True, min_periods=3).mean()
        ax.plot(d["cycle"], d[s], color=col, ls=dash, lw=0.6, alpha=0.18, zorder=2)
        ax.plot(d["cycle"], smooth, color=col, ls=dash, lw=2.4, zorder=3,
                solid_capstyle="round")
        # engine tag at the end of its own trace: the longest one is placed
        # inside the axes so it cannot be clipped
        last = u == life.idxmax()
        ax.annotate(f"e{u}", (d["cycle"].iloc[-1], smooth.iloc[-1]),
                    xytext=(-6, 6) if last else (5, 6), textcoords="offset points",
                    ha="right" if last else "left", va="bottom",
                    fontsize=9, color=col, zorder=4,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white",
                              ec="none", alpha=0.72))
    ax.set_title(f"{'Rises' if corr[s] < 0 else 'Falls'} toward failure  ·  "
                 f"|corr| with RUL = {abs(corr[s]):.2f}", loc="left")
    ax.set_ylabel(s)
    ax.margins(x=0.02, y=0.09)
    ax.set_xlim(0, life.max() * 1.03)
    ax.set_axisbelow(True)

for a in axes[1, :]:
    a.set_xlabel("Life cycle (cycles since first record)")

handles = ([Line2D([], [], color="#404040", lw=2.4, ls=dash,
                   label=f"Engine {u} · {int(life[u])} cycles")
            for u, dash in zip(engines, ENGINE_DASH)]
           + [Line2D([], [], color="#404040", lw=0.8, alpha=0.45,
                     label="raw record (faint)"),
              Line2D([], [], color="#404040", lw=2.4,
                     label=f"rolling mean, {WIN}-cycle window (bold)"),
              Line2D([], [], color="#A6A6A6", lw=2.6,
                     label=f"running-engine mean · stops below {MIN_ALIVE} engines alive")])
fig.suptitle(f"{len(engines)} of {train['unit'].nunique()} engines "
             f"({int(life.min())}-{int(life.max())} cycles) move the same way — "
             f"the trend clears the sensor noise",
             x=0.077, y=0.972, ha="left", fontsize=13, fontweight="bold", color="#1A1A1A")
fig.text(0.077, 0.930, f"faint = raw records  ·  bold = {WIN}-cycle centred rolling mean  ·  "
         "grey = fleet mean  ·  colour marks the sensor, dashing marks the engine",
         ha="left", fontsize=9, color="#555555")
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.012),
           ncol=4, frameon=False, columnspacing=1.4, handlelength=2.6)


## Fleet trend

In [ ]:
BIN = 10         # RUL bin width (cycles)
MIN_ENGINES = 20  # no mean is drawn where fewer engines than this remain
bins = pd.cut(train["RUL"], bins=range(0, int(train["RUL"].max()) + 20, BIN),
              include_lowest=True)
grp = train.groupby(bins, observed=True)
n_eng = grp["unit"].nunique()  # engines contributing to each RUL bin
x = np.array([b.mid for b in n_eng.index])
keep = n_eng.to_numpy() >= MIN_ENGINES
thin_from = float(n_eng.index[~keep][0].left) if (~keep).any() else float(x.max() + BIN / 2)
x_end = thin_from + BIN / 2  # padding only; the excluded tail is stated in words

fig, axes = plt.subplots(2, 2, figsize=(11, 7.4), dpi=120,
                         sharex=True, sharey=True)
fig.subplots_adjust(left=0.085, right=0.985, top=0.865, bottom=0.115,
                    hspace=0.34, wspace=0.10)

for ax, s in zip(axes.ravel(), TOP):
    col = SENSOR_COLORS[s]
    z = (train[s] - train[s].mean()) / train[s].std()
    gz = z.groupby(bins, observed=True)
    mean = gz.mean().to_numpy()
    q1, q3 = gz.quantile(0.25).to_numpy(), gz.quantile(0.75).to_numpy()
    ax.fill_between(x[keep], q1[keep], q3[keep], color=col, alpha=0.18, lw=0, zorder=2)
    ax.plot(x[keep], mean[keep], color=col, lw=2.2, zorder=3)
    ax.set_title(f"{s}  ·  |corr| with RUL = {abs(corr[s]):.2f}  ·  "
                 f"{'rises' if corr[s] < 0 else 'falls'} toward failure", loc="left")
    ax.set_axisbelow(True)
    ax.set_xlim(x_end, -BIN / 2)

axes[0, 0].annotate(f"RUL > {thin_from:.0f}\n<{MIN_ENGINES} engines\nnot shown",
                    xy=(0.04, 0.95), xycoords="axes fraction", fontsize=9,
                    color="#595959", ha="left", va="top")
for a in axes[:, 0]:
    a.set_ylabel("sensor value (z-score)")
for a in axes[1, :]:
    a.set_xlabel("Remaining useful life (cycles until failure)")

fig.suptitle("All four leading sensors drift steadily toward failure",
             x=0.085, y=0.972, ha="left", fontsize=13, fontweight="bold", color="#1A1A1A")
fig.text(0.085, 0.932, f"point = mean of a 10-cycle RUL bin  ·  band = IQR (25-75%) "
         f"across engines  ·  shared z-scale  ·  RUL > {thin_from:.0f} not shown: "
         f"fewer than {MIN_ENGINES} engines left", ha="left", fontsize=9, color="#555555")


## Model input

The model gets one row per recorded cycle, with the 14 kept sensors as features. The target is the remaining useful life of that cycle.

In [ ]:
ENGINES = np.sort(train["unit"].unique())
HOLDOUT = ENGINES[::5]  # every fifth engine: 20 units never appear in fitting

X = train[KEEP_SENSORS].to_numpy(float)
y = train["RUL"].to_numpy(float)
held = train["unit"].isin(HOLDOUT).to_numpy()

print(f"fitting rows: {int((~held).sum())}")
print(f"held-out rows: {int(held.sum())}")
print(f"held-out engines: {len(HOLDOUT)}")


## Baseline model

Plain least squares on the 14 sensors, no tuning. Fitted only to the 80 fitting engines, scored only on the 20 held-out ones. The yardstick is `always guess the average`.

In [ ]:
mean = X[~held].mean(axis=0)
std = X[~held].std(axis=0)
Z = (X - mean) / std

A = np.column_stack([np.ones(len(X)), Z])
coef = np.linalg.lstsq(A[~held], y[~held], rcond=None)[0]

pred = A[held] @ coef
rmse = np.sqrt(np.mean((pred - y[held]) ** 2))

mean_y = y[~held].mean()
baseline = np.sqrt(np.mean((mean_y - y[held]) ** 2))

print(f"held-out RMSE: {rmse:.1f} cycles")
print(f"always-guess-the-average RMSE: {baseline:.1f} cycles")

## Score on the real test set

The test engines stop part-way through life, so we predict once from each engine's last recorded cycle and compare with the 100 true values in `RUL_FD001.txt`; this is the number the README quotes. Predictions below zero are raised to zero because negative remaining life is impossible.

In [ ]:
last = test.sort_values(["unit", "cycle"]).groupby("unit").tail(1)
Zt = (last[KEEP_SENSORS].to_numpy(float) - mean) / std
pred_test = np.clip(np.column_stack([np.ones(len(last)), Zt]) @ coef, 0, None)
truth = rul.to_numpy(float)

rmse_test = np.sqrt(np.mean((pred_test - truth) ** 2))
average = y[~held].mean()
baseline_test = np.sqrt(np.mean((average - truth) ** 2))

print(f"test RMSE: {rmse_test:.1f} cycles")
print(f"always-guess-the-average RMSE on test: {baseline_test:.1f} cycles")
print(f"mean prediction {pred_test.mean():.1f} vs mean truth {truth.mean():.1f} cycles")

## Where the model fails

The model leans high: on average it over-predicts by 18.8 cycles, and the lean is about the same from fresh engines to dying ones, so it is a bias, not a slope error. That bias accounts for 39% of the squared error; the other 61% is scatter, which is what the second panel shows. Both panels show the same 100 test engines as points, not lines.

In [ ]:
err = pred_test - truth
bias = float(err.mean())
lo = -8.0  # small margin so the clipped-at-zero predictions stay whole
span = float(max(truth.max(), pred_test.max())) * 1.06
y_lo, y_hi = float(err.min()) - 8, float(err.max()) + 8

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.9, 5.5), dpi=120)
fig.subplots_adjust(left=0.075, right=0.985, top=0.815, bottom=0.135, wspace=0.28)

# Panel 1: the same point cloud the RMSE is computed on, against the perfect diagonal.
ax1.plot([0, span], [0, span], color="#8C8C8C", lw=1.0, ls=(0, (4, 2.4)),
         label="perfect prediction", zorder=1)
ax1.scatter(truth, pred_test, s=26, color="#0072B2", alpha=0.72, linewidths=0, zorder=2)
ax1.set_xlim(lo, span)
ax1.set_ylim(lo, span)
ax1.set_aspect("equal", adjustable="box")
ax1.set_xlabel("true RUL (cycles)")
ax1.set_ylabel("predicted RUL (cycles)")
ax1.set_title(f"Predicted against true RUL  ·  RMSE {rmse_test:.1f} cycles", loc="left")
ax1.legend(loc="upper left", frameon=False, handlelength=2.2, borderaxespad=0.6)
ax1.set_axisbelow(True)

# Panel 2: error only. Dashed band = the scored RMSE, orange line = the lean.
ax2.axhspan(-rmse_test, rmse_test, color="#0072B2", alpha=0.08, lw=0, zorder=0)
for edge in (-rmse_test, rmse_test):
    ax2.axhline(edge, color="#0072B2", lw=0.9, ls=(0, (4, 3)), zorder=1)
ax2.axhline(0, color="#707070", lw=1.0, zorder=1)
ax2.axhline(bias, color="#D55E00", lw=1.5, zorder=3)
ax2.scatter(truth, err, s=26, color="#0072B2", alpha=0.72, linewidths=0, zorder=2)
ax2.set_xlim(lo, span)
ax2.set_ylim(y_lo, y_hi)
ax2.set_xlabel("true RUL (cycles)")
ax2.set_ylabel("error: predicted − true (cycles)")
ax2.set_title(f"Mean error {bias:+.1f}  ·  band ±{rmse_test:.1f} cycles", loc="left")
ax2.set_axisbelow(True)

fig.suptitle(f"Beats always guessing the average ({rmse_test:.1f} vs {baseline_test:.1f} cycles), "
             f"but leans about {bias:.0f} cycles high",
             x=0.075, y=0.955, ha="left", fontsize=12.5, fontweight="bold", color="#1A1A1A")


## Conclusions

(written by the author)